# Notebook 4: Fine-Tuning

**Goal:** Fine-tune the sentence-transformer model on Food.com interaction data and measure whether domain adaptation improves retrieval quality.

**Research question:** Does domain-specific fine-tuning on food interaction data improve recommendation quality, even with proxy labels (ratings vs. actual orders)?

**Requires:** GPU runtime. Go to *Runtime → Change runtime type → T4 GPU*.

In [1]:
import torch
assert torch.cuda.is_available(), "⚠️  This notebook requires GPU. Go to Runtime > Change runtime type > GPU"
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

✓ GPU: Tesla T4


In [3]:
# ============================================================
# SETUP
# ============================================================
import os, sys

REPO = "Embedding-Based-Recommender"
GITHUB_USER = "IldarRakiev"

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab'
BASE = '/kaggle/working' if ENV == 'kaggle' else '/content'
REPO_DIR = f'{BASE}/{REPO}'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/{GITHUB_USER}/{REPO}.git {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull -q')

os.system('pip install -q sentence-transformers faiss-cpu pandas pyarrow matplotlib seaborn umap-learn plotly scikit-learn tqdm')
sys.path.insert(0, f'{REPO_DIR}/src')
print(f"Environment: {ENV} | Repo: {REPO_DIR}")
print("Setup complete")

Cloning into '/kaggle/working/Embedding-Based-Recommender'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.0 MB/s eta 0:00:00
Environment: kaggle | Repo: /kaggle/working/Embedding-Based-Recommender
Setup complete


In [7]:
# ============================================================
# DATA PATHS
# ============================================================
import os, glob as _glob

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab' if os.path.exists('/content') else 'local'

if ENV == 'local':
    SYNTHETIC_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data', 'synthetic')
else:
    SYNTHETIC_DIR = os.path.join(REPO_DIR, 'data', 'synthetic')

OUTPUT_DIR = '/kaggle/working/processed' if ENV == 'kaggle' else SYNTHETIC_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"SYNTHETIC_DIR = {SYNTHETIC_DIR}")
print(f"OUTPUT_DIR    = {OUTPUT_DIR}")

SYNTHETIC_DIR = /kaggle/working/Embedding-Based-Recommender/data/synthetic
OUTPUT_DIR    = /kaggle/working/processed


In [5]:
from text_builders import dish_to_rich_text
from embedding_model import EmbeddingModel
from utils import evaluate_all, load_run_results_json, plot_embeddings_umap
import pandas as pd
import numpy as np
np.random.seed(42)

In [9]:
import json, faiss, os

dishes = pd.read_parquet(f'{SYNTHETIC_DIR}/dishes.parquet')
train = pd.read_parquet(f'{SYNTHETIC_DIR}/interactions_train.parquet')
test = pd.read_parquet(f'{SYNTHETIC_DIR}/interactions_test.parquet')

dish_id_to_idx = {did: i for i, did in enumerate(dishes['id'])}
idx_to_dish_id = {i: did for i, did in enumerate(dishes['id'])}

# Use best text config from NB3
id_to_text = {
    row['id']: dish_to_rich_text(row.to_dict(), tags=row.get('tag_list', []),
        include_recipe=False, include_macro_tokens=False, include_ratios=True, include_ingredients=True)
    for _, row in dishes.iterrows()
}

_results_candidates = [
    os.path.join(OUTPUT_DIR, "results.json"),
    os.path.join(SYNTHETIC_DIR, "../results.json"),
]
if os.path.isdir("/kaggle/input"):
    import glob as _glob

    _results_candidates.extend(_glob.glob("/kaggle/input/**/results.json", recursive=True))

all_results = load_run_results_json(*_results_candidates)
if not all_results:
    print("WARN: results.json missing — run NB2/3 to produce it or add as Kaggle Input. Starting empty dict.")
print(f"Loaded {len(dishes):,} dishes | results keys: {len(all_results)}")

Loaded 4,939 dishes | results keys: 2


## 1. Training pair construction

We use **`MultipleNegativesRankingLoss`** with **two texts per example** `(anchor, positive)` only.

- **Positive pair:** two different dishes that the **same user** ordered or favorited in the **train** split (co-preference).
- **Negatives:** all other `(anchor, positive)` rows in the **same mini-batch** act as in-batch negatives (standard MNRL). No separate FAISS hard-negative step — the third sentence in a triplet was **not** used correctly by MNRL before.

This matches the loss API and usually trains more stably than mis-wired triplets.

In [10]:
from sentence_transformers import SentenceTransformer, InputExample

# Load pretrained model (same checkpoint as NB2/NB3)
base_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

# If this cell is run out-of-order (or after a kernel restart), reload data/texts.
try:
    dishes  # noqa: F821
    id_to_text  # noqa: F821
except NameError:
    import pandas as pd

    dishes = pd.read_parquet(f"{SYNTHETIC_DIR}/dishes.parquet")
    train = pd.read_parquet(f"{SYNTHETIC_DIR}/interactions_train.parquet")
    test = pd.read_parquet(f"{SYNTHETIC_DIR}/interactions_test.parquet")

    dish_id_to_idx = {did: i for i, did in enumerate(dishes["id"])}
    idx_to_dish_id = {i: did for i, did in enumerate(dishes["id"])}

    id_to_text = {
        row["id"]: dish_to_rich_text(
            row.to_dict(),
            tags=row.get("tag_list", []),
            include_recipe=False,
            include_macro_tokens=False,
            include_ratios=True,
            include_ingredients=True,
        )
        for _, row in dishes.iterrows()
    }

# Frozen snapshot of pretrained embeddings for UMAP comparison later (before .fit mutates weights)
print("Encoding pretrained dish texts (for UMAP baseline + same row order as dishes)...")
texts_list = [id_to_text[rid] for rid in dishes["id"] if rid in id_to_text]
ids_list = [rid for rid in dishes["id"] if rid in id_to_text]

pretrained_embs = base_model.encode(
    texts_list,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(f"pretrained_embs: {pretrained_embs.shape}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding pretrained dish texts (for UMAP baseline + same row order as dishes)...


Batches:   0%|          | 0/78 [00:00<?, ?it/s]

pretrained_embs: (4939, 768)


In [11]:
# Train positives = orders + favorites from train set
user_train_positives = (
    train[train["interaction_type"].isin(["order", "favorite"])]
    .groupby("user_id")["dish_id"]
    .apply(list)
    .to_dict()
)

MAX_PAIRS = 80_000
PAIRS_PER_USER = 25
rng = np.random.default_rng(42)

train_examples: list[InputExample] = []
user_order = list(user_train_positives.keys())
rng.shuffle(user_order)

for user_id in user_order:
    pos_set = {d for d in user_train_positives[user_id] if d in id_to_text}
    if len(pos_set) < 5:
        continue
    ids = np.array(list(pos_set), dtype=object)
    n_draws = min(PAIRS_PER_USER, max(1, len(ids) * 3))
    for _ in range(n_draws):
        a, b = rng.choice(ids, size=2, replace=False)
        train_examples.append(InputExample(texts=[id_to_text[a], id_to_text[b]]))
        if len(train_examples) >= MAX_PAIRS:
            break
    if len(train_examples) >= MAX_PAIRS:
        break

print(f"Training pairs (MNRL anchor/positive): {len(train_examples):,}")
ex = train_examples[0]
print("\nExample pair:")
print(f"  A: {ex.texts[0][:120]}...")
print(f"  B: {ex.texts[1][:120]}...")

Training pairs (MNRL anchor/positive): 27,160

Example pair:
  A: DISH_NAME: Crispy Quinoa Salad with Lemon Dressing
DESCRIPTION: This crisp and refreshing salad features crunchy vegetab...
  B: DISH_NAME: Chia Seed Mango Bars
DESCRIPTION: These no-bake bars combine chia seeds and tropical mango for a chewy and nu...


## 2. Fine-Tuning

In [12]:
from sentence_transformers import losses
from torch.utils.data import DataLoader

EPOCHS = 4
BATCH_SIZE = 64
steps_per_epoch = max(1, len(train_examples) // BATCH_SIZE)
WARMUP_STEPS = min(500, steps_per_epoch)

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=BATCH_SIZE,
    drop_last=True,
)
train_loss = losses.MultipleNegativesRankingLoss(base_model)

print(
    f"Fine-tuning (MNRL): {len(train_examples):,} pairs | {EPOCHS} epochs | "
    f"batch={BATCH_SIZE} | drop_last=True"
)
base_model.fit(
    use_amp=True,
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path="models/food-recsys-finetuned",
    show_progress_bar=True,
)
print("Fine-tuning complete")

Fine-tuning (MNRL): 27,160 pairs | 4 epochs | batch=64 | drop_last=True


Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,4.906417


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete


## 2b. Fine-Tuning with Hard Negatives (TripletLoss)

`MultipleNegativesRankingLoss` relies on *random* in-batch negatives, which can be too easy for this synthetic setup.

Here we explicitly mine **hard negatives**:
- pick a dish from the **same `cuisine_cluster`** as the anchor
- ensure the user **did not** order/favorite it
- prefer dishes that are **cosine-similar to the anchor** within that cluster (hard contrast)

Then we fine-tune with `TripletLoss(anchor, positive, hard_negative)`.

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import torch
from sentence_transformers import losses
from torch.utils.data import DataLoader

from hard_negative_mining import (
    HardNegativeMiningConfig,
    build_triplets_same_user_with_hard_negatives,
)

# ------------------------------------------------------------
# CUDA / VRAM cleanup before the 2nd training run
# ------------------------------------------------------------
# Running two fine-tunes back-to-back often OOMs on Colab/Kaggle T4s because
# sentence-transformers keeps the previous model + training objects alive.
import gc

for name in ("train_dataloader", "train_loss"):
    if name in globals():
        del globals()[name]

# Drop the MNRL-finetuned model from GPU memory (we reload pretrained for triplet anyway)
if "base_model" in globals():
    try:
        base_model.to("cpu")
    except Exception:
        pass
    del base_model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    free_b, total_b = torch.cuda.mem_get_info()
    print(f"CUDA mem after cleanup: {free_b/1e9:.2f} / {total_b/1e9:.2f} GB free")

# IMPORTANT: start triplet fine-tuning from pretrained weights (not MNRL weights)
triplet_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

# Mine hard-negative triplets using pretrained embedding space (computed earlier)
# pretrained_embs aligns with dishes['id'] order used for texts_list.
mining_cfg = HardNegativeMiningConfig(candidate_pool=50)

text_triplets = build_triplets_same_user_with_hard_negatives(
    interactions_train_df=train,
    id_to_text=id_to_text,
    dishes_df=dishes,
    dish_embeddings=pretrained_embs.astype(np.float32),
    max_triplets=80_000,
    triplets_per_user=25,
    seed=42,
    cfg=mining_cfg,
)

triplet_examples = [InputExample(texts=[a, p, n]) for (a, p, n) in text_triplets]
print(f"Training triplets (TripletLoss a/p/hard_neg): {len(triplet_examples):,}")

TRIPLET_EPOCHS = 3
TRIPLET_BATCH = 16
steps_per_epoch = max(1, len(triplet_examples) // TRIPLET_BATCH)
TRIPLET_WARMUP = min(500, steps_per_epoch)

triplet_dataloader = DataLoader(
    triplet_examples,
    shuffle=True,
    batch_size=TRIPLET_BATCH,
    drop_last=True,
)
triplet_loss = losses.TripletLoss(model=triplet_model)

print(
    f"Fine-tuning (TripletLoss + hard negatives): {len(triplet_examples):,} triplets | "
    f"{TRIPLET_EPOCHS} epochs | batch={TRIPLET_BATCH} | drop_last=True"
)
triplet_model.fit(
    use_amp=True,
    train_objectives=[(triplet_dataloader, triplet_loss)],
    epochs=TRIPLET_EPOCHS,
    warmup_steps=TRIPLET_WARMUP,
    output_path="models/food-recsys-triplet-hardneg",
    show_progress_bar=True,
)
print("Triplet hard-negative fine-tuning complete")

# Free VRAM before evaluation / next steps
for name in ("triplet_dataloader", "triplet_loss"):
    if name in globals():
        del globals()[name]

try:
    triplet_model.to("cpu")
except Exception:
    pass
del triplet_model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass


In [ ]:
import torch
import gc

# Extra cleanup before loading another SentenceTransformer + encoding pass
for name in ("triplet_finetuned_model",):
    if name in globals():
        try:
            triplet_finetuned_model.to("cpu")
        except Exception:
            pass
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

# ============================================================
# Evaluate Triplet (hard negatives) model
# ============================================================
triplet_finetuned_model = SentenceTransformer('models/food-recsys-triplet-hardneg')

triplet_embs = triplet_finetuned_model.encode(
    texts_list,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

triplet_index = faiss.IndexFlatIP(triplet_embs.shape[1])
triplet_index.add(triplet_embs)

metrics_triplet = evaluate_retrieval(triplet_index, triplet_embs)
all_results['finetuned_triplet_hardneg'] = metrics_triplet

print("=== Fine-tuned variants comparison ===")
comparison2 = pd.DataFrame({
    'Fine-tuned (MNRL)': metrics_finetuned,
    'Fine-tuned (Triplet hardneg)': metrics_triplet,
}).T
print(comparison2[['P@5', 'P@10', 'NDCG@10', 'MRR']].round(4))

print(f"\nΔ P@10 (Triplet - MNRL): {metrics_triplet.get('P@10', 0) - metrics_finetuned.get('P@10', 0):+.4f}")

## 3. Evaluate: Pretrained vs Fine-tuned

In [13]:
def evaluate_retrieval(index, embeddings, ks=None):
    if ks is None: ks = [5, 10, 20]
    user_positives = (
        test[test['interaction_type'].isin(['order', 'favorite'])]
        .groupby('user_id')['dish_id'].apply(set).to_dict()
    )
    results = []
    for user_id, pos_dishes in user_positives.items():
        pos_dishes = {d for d in pos_dishes if d in dish_id_to_idx}
        if len(pos_dishes) < 5: continue
        query_dish = list(pos_dishes)[0]
        relevant = pos_dishes - {query_dish}
        query_idx = dish_id_to_idx[query_dish]
        scores, indices = index.search(embeddings[query_idx:query_idx+1], max(ks) + 1)
        recommended = [idx_to_dish_id[i] for i in indices[0] if i >= 0 and idx_to_dish_id.get(i) != query_dish]
        results.append(evaluate_all(recommended, relevant, ks=ks))
    return pd.DataFrame(results).mean().to_dict() if results else {}

finetuned_model = SentenceTransformer('models/food-recsys-finetuned')
texts_list = [id_to_text[did] for did in dishes['id'] if did in id_to_text]
ids_list = [did for did in dishes['id'] if did in id_to_text]

finetuned_embs = finetuned_model.encode(texts_list, batch_size=64, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)

finetuned_index = faiss.IndexFlatIP(finetuned_embs.shape[1])
finetuned_index.add(finetuned_embs)

metrics_finetuned = evaluate_retrieval(finetuned_index, finetuned_embs)
all_results['finetuned'] = metrics_finetuned

metrics_best_pretrained = all_results.get('full_improved', {})
comparison = pd.DataFrame({
    'Best Pretrained': metrics_best_pretrained,
    'Fine-tuned': metrics_finetuned,
}).T
print("=== Pretrained vs Fine-tuned ===")
print(comparison[['P@5', 'P@10', 'NDCG@10', 'MRR']].round(4))

delta_p10 = metrics_finetuned.get('P@10', 0) - metrics_best_pretrained.get('P@10', 0)
print(f"\nΔ P@10: {delta_p10:+.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/78 [00:00<?, ?it/s]

=== Pretrained vs Fine-tuned ===
                    P@5    P@10  NDCG@10     MRR
Best Pretrained     NaN     NaN      NaN     NaN
Fine-tuned       0.0013  0.0015   0.0016  0.0053

Δ P@10: +0.0015


## 4. UMAP: Pretrained vs Fine-tuned

In [ ]:
SAMPLE = 2000
sample_idx = np.random.choice(len(pretrained_embs), SAMPLE, replace=False)

id_to_tags = dict(zip(dishes['id'], dishes['tag_list']))
sample_ids = [ids_list[i] for i in sample_idx]

labels = []
for rid in sample_ids:
    tags = id_to_tags.get(rid, [])
    label = next((t for t in tags if t in ['breakfast','lunch','dinner','snacks']), 'other')
    labels.append(label)

fig, axes = __import__('matplotlib.pyplot', fromlist=['subplots']).subplots(1, 2, figsize=(16, 6))

import matplotlib.pyplot as plt
import umap

for ax, embs, title in [
    (axes[0], pretrained_embs[sample_idx], 'Pretrained Embeddings'),
    (axes[1], finetuned_embs[sample_idx], 'Fine-tuned Embeddings'),
]:
    coords = umap.UMAP(n_components=2, random_state=42, metric='cosine').fit_transform(embs)
    unique = list(set(labels))
    cmap = plt.cm.get_cmap('tab10', len(unique))
    for i, lab in enumerate(unique):
        mask = [j for j, l in enumerate(labels) if l == lab]
        ax.scatter(coords[mask, 0], coords[mask, 1], s=6, alpha=0.5, color=cmap(i), label=lab)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Fine-tuned + all improvements combined
# (fine-tuned model already uses best text config)
all_results['finetuned_full'] = metrics_finetuned

with open(f'{OUTPUT_DIR}/results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

# Save embeddings for reuse in other notebooks
np.save(f'{OUTPUT_DIR}/embeddings_finetuned.npy', finetuned_embs)
np.save(f'{OUTPUT_DIR}/embeddings_finetuned_triplet_hardneg.npy', triplet_embs)
print("✓ Saved")